# Bearing Fault Classification — EDA & Machine Learning

This notebook takes a CSV of statistical vibration features (`max`, `min`, `mean`, `sd`, `rms`,
`skewness`, `kurtosis`, `crest`, `form`) and a label column (`fault`) describing the bearing
condition (e.g. `Ball_007_1`, `Normal`, `IR_014_2`, ...), and:

1. Loads and explores the data (EDA)
2. Prepares it for machine learning
3. Trains several supervised classification models and compares them
4. Takes the **best** model and validates it properly with **k-fold cross-validation**
5. Produces the key graphs and metrics needed to judge the result
6. Ends with a plain-language summary

> **Before running:** put your CSV file in the same folder as this notebook and set
> `DATA_PATH` in the cell below (default: `data.csv`).


In [ ]:
# --- Setup: imports & config ---
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42

# ---- CONFIG: change this if your file has a different name/path ----
DATA_PATH = "data.csv"
TARGET_COLUMN = "fault"   # the column we want to predict


## 1. Load the data

In [ ]:
df = pd.read_csv("feature_time_48k_2048_load_1.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
# Basic sanity checks
print("Missing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
# Drop exact duplicate rows (if any) and rows with missing target
df = df.drop_duplicates()
df = df.dropna(subset=[TARGET_COLUMN])
print("Shape after cleaning:", df.shape)


## 2. Exploratory Data Analysis (EDA)

We look at:
- How many classes (fault types) we have and whether they're balanced
- The distribution of each numeric feature
- Whether features differ across fault classes
- How features correlate with each other


### 2.1 Class balance

In [ ]:
class_counts = df[TARGET_COLUMN].value_counts()
print(class_counts)

plt.figure(figsize=(10, 5))
sns.barplot(x=class_counts.index, y=class_counts.values, palette="viridis")
plt.title("Number of samples per fault class")
plt.xlabel("Fault class")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### 2.2 Feature distributions

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric feature columns:", numeric_cols)

df[numeric_cols].describe().T


In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(numeric_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color="steelblue")
    axes[i].set_title(f"Distribution of {col}")

for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


### 2.3 Feature spread by fault class

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df, x=TARGET_COLUMN, y=col, ax=axes[i], palette="Set2")
    axes[i].set_title(f"{col} by fault class")
    axes[i].tick_params(axis="x", rotation=45)

for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


### 2.4 Correlation between features

In [ ]:
plt.figure(figsize=(9, 7))
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlation matrix")
plt.tight_layout()
plt.show()


## 3. Prepare data for machine learning

- Encode the text fault labels as numbers
- Split into train/test sets (stratified, so each fault class is represented proportionally)
- Scale the features (important for models like Logistic Regression, KNN and SVM)


In [ ]:
X = df[numeric_cols].copy()
y_raw = df[TARGET_COLUMN].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_names = label_encoder.classes_
print("Classes:", list(class_names))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


## 4. Try several supervised learning models

We train a handful of common classifiers with default-ish settings and compare them
on the held-out test set using **accuracy** and **weighted F1-score** (F1 is a better
measure than accuracy when classes are imbalanced).


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "SVM (RBF)": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "F1 (weighted)": f1_score(y_test, preds, average="weighted"),
        "Precision (weighted)": precision_score(y_test, preds, average="weighted"),
        "Recall (weighted)": recall_score(y_test, preds, average="weighted"),
    })

results_df = pd.DataFrame(results).sort_values("F1 (weighted)", ascending=False).reset_index(drop=True)
results_df


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=results_df, x="F1 (weighted)", y="Model", palette="crest")
plt.title("Model comparison (test set, weighted F1-score)")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()


## 5. Pick the best model and validate it with k-fold cross-validation

We take the model with the highest weighted F1-score above and re-evaluate it more
rigorously using **5-fold stratified cross-validation** on the full training data.
This gives a more trustworthy estimate of how well the model generalizes, instead of
relying on a single train/test split.


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
print(f"Best model based on test F1-score: {best_model_name}")


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_accuracy = cross_val_score(best_model, X_train_scaled, y_train, cv=cv, scoring="accuracy")
cv_f1 = cross_val_score(best_model, X_train_scaled, y_train, cv=cv, scoring="f1_weighted")

print(f"5-Fold CV Accuracy: {cv_accuracy.mean():.4f}  (+/- {cv_accuracy.std():.4f})")
print(f"5-Fold CV F1 (weighted): {cv_f1.mean():.4f}  (+/- {cv_f1.std():.4f})")

cv_scores_df = pd.DataFrame({
    "Fold": [f"Fold {i+1}" for i in range(len(cv_accuracy))] * 2,
    "Score": np.concatenate([cv_accuracy, cv_f1]),
    "Metric": ["Accuracy"] * len(cv_accuracy) + ["F1 (weighted)"] * len(cv_f1),
})


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=cv_scores_df, x="Metric", y="Score", palette="pastel")
sns.stripplot(data=cv_scores_df, x="Metric", y="Score", color="black", size=7, jitter=True)
plt.title(f"{best_model_name}: 5-fold cross-validation scores")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()


## 6. Final evaluation on the test set

We fit the best model on the full training set and evaluate it on the untouched test
set, with a confusion matrix and a full classification report.


In [ ]:
best_model.fit(X_train_scaled, y_train)
final_preds = best_model.predict(X_test_scaled)

final_accuracy = accuracy_score(y_test, final_preds)
final_f1 = f1_score(y_test, final_preds, average="weighted")

print(f"Final test accuracy: {final_accuracy:.4f}")
print(f"Final test F1 (weighted): {final_f1:.4f}")
print()
print("Classification report:")
print(classification_report(y_test, final_preds, target_names=class_names))


In [ ]:
cm = confusion_matrix(y_test, final_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title(f"Confusion matrix — {best_model_name}")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### 6.1 Which features matter most? (if supported by the model)

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=numeric_cols).sort_values(ascending=False)

    plt.figure(figsize=(8, 5))
    sns.barplot(x=importances.values, y=importances.index, palette="mako")
    plt.title(f"Feature importance — {best_model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
elif hasattr(best_model, "coef_"):
    coef = pd.Series(np.abs(best_model.coef_).mean(axis=0), index=numeric_cols).sort_values(ascending=False)

    plt.figure(figsize=(8, 5))
    sns.barplot(x=coef.values, y=coef.index, palette="mako")
    plt.title(f"Average |coefficient| by feature — {best_model_name}")
    plt.xlabel("Average absolute coefficient")
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_model_name} does not expose feature importances or coefficients directly.")


## 7. Summary

- **Data**: statistical vibration features (`max`, `min`, `mean`, `sd`, `rms`, `skewness`,
  `kurtosis`, `crest`, `form`) used to predict the bearing `fault` class.
- **Models tried**: Logistic Regression, K-Nearest Neighbors, Decision Tree, Random Forest,
  Gradient Boosting, and SVM (RBF kernel).
- **Best model**: printed above as `best_model_name`, selected using weighted F1-score on
  a held-out test set.
- **Validation**: the best model was re-checked with 5-fold stratified cross-validation to
  make sure its performance is consistent and not a lucky split, then evaluated one more
  time on the untouched test set (confusion matrix + classification report).
- **How to read the results**:
  - High accuracy/F1 (close to 1.0) and small standard deviation across folds → the model
    is reliable.
  - The confusion matrix shows exactly which fault classes get confused with each other —
    useful for deciding if more data or better features are needed for those specific
    classes.
  - The feature importance chart shows which vibration statistics drive the model's
    decisions the most.

Run the cell below to print a short, auto-generated recap using the actual numbers from
this run.


In [ ]:
print("="*60)
print("SUMMARY")
print("="*60)
print(f"Best performing model : {best_model_name}")
print(f"Test accuracy          : {final_accuracy:.4f}")
print(f"Test F1 (weighted)     : {final_f1:.4f}")
print(f"5-fold CV accuracy     : {cv_accuracy.mean():.4f} (+/- {cv_accuracy.std():.4f})")
print(f"5-fold CV F1 (weighted): {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})")
print("="*60)
